Example script for mutual information computations from data

There are two possibilities:
1) Running this individually for unalligned z-stacks before and after major z-stack correction, both pieces individually tracked
2) Running this fully on the tracked alligned z-stacks

The code doesn't really change. The only thing that changes is the path directories for the files below.

In the paper, we used the former, but observed no qualitative difference with the latter as well.

The code provided is for the second case listed above for ease of use. However, the code can be easily amended for the first use case as well.

As noted in the README, the execution requires access to the package NPEET

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.animation import FuncAnimation, FFMpegWriter
from scipy.signal import savgol_filter
import numpy as np
import pickle
from scipy import stats
from npeet import entropy_estimators as ee

In [ ]:
%matplotlib inline

In [ ]:
name_exp="s3"
#There are three different experiments: s1, s2, s3
#The code is the same for all three

# Load the data

In [ ]:
gfp_profil = pd.read_csv("processed_experiments/gfp_profils_all_adjusted/"+name_exp+"/gfp_profil_time_corrected_cleaned_"+name_exp+"_sigma10.csv")

In [ ]:
gfp_profil

In [ ]:
with open('processed_experiments/relevant_dataframes_all_adjusted/'+name_exp+'/dataframes_'+name_exp+'_tracked_new_FDI.pkl', 'rb') as f:
    all_tracks_processed = pickle.load(f)

with open('processed_experiments/relevant_dataframes_all_adjusted/'+name_exp+'/dataframes_'+name_exp+'_tracked_new_filtered_FDI.pkl', 'rb') as f:
    all_tracks_processed_and_filtered = pickle.load(f)


In [ ]:
print(all_tracks_processed_and_filtered.keys())

In [ ]:
track_lengths = {track_id: len(df) for track_id, df in all_tracks_processed_and_filtered.items()}
sorted_lengths = sorted(track_lengths.items(), key=lambda x: x[1], reverse=True)
for track_id, length in sorted_lengths:
    print(f"Track ID: {track_id}, Length: {length}")

In [ ]:
all_tracks_processed_and_filtered[11442]

In [ ]:
for track_id in all_tracks_processed_and_filtered.keys():
    curr_track_data = all_tracks_processed_and_filtered[track_id]
    times = curr_track_data['time']
    nucleus_norm = curr_track_data['channel_2_norm']
    z_stack_mean = gfp_profil['mean']
    z_stack_std = gfp_profil['std']
    # Calculate nucleus_raw by multiplying nucleus_norm with z_stack_mean 
    # Only for the times that exist in curr_track_data

    nucleus_neigh_norm_mean = curr_track_data['neighs_mean_norm']
    nucleus_neigh_norm_std = curr_track_data['neighs_std_norm']

    nucleus_zscore = (nucleus_norm - np.array([z_stack_mean[t-1] for t in times]) ) / np.array([z_stack_std[t-1] for t in times]) 
    neigh_zscore = (nucleus_neigh_norm_mean - np.array([z_stack_mean[t-1] for t in times]) ) / np.array([z_stack_std[t-1] for t in times]) 
    neigh_zscore_std = (nucleus_neigh_norm_std - np.array([z_stack_std[t-1] for t in times]) ) / np.array([z_stack_std[t-1] for t in times]) 
   
    #savgol filter to smooth the data
    nucleus_norm_smoothed = savgol_filter(nucleus_norm, window_length=min(10, len(nucleus_norm)), polyorder=min(2, len(nucleus_norm)-1))
    nucleus_neigh_norm_mean_smoothed = savgol_filter(nucleus_neigh_norm_mean, window_length=min(10, len(nucleus_neigh_norm_mean)), polyorder=min(2, len(nucleus_neigh_norm_mean)-1))

    # Save smoothed data back to dataframe
    curr_track_data['nucleus_norm_smoothed'] = nucleus_norm_smoothed
    curr_track_data['neigh_norm_smoothed'] = nucleus_neigh_norm_mean_smoothed
    FDI_calcs=(nucleus_norm_smoothed-nucleus_neigh_norm_mean_smoothed)/(nucleus_norm_smoothed+nucleus_neigh_norm_mean_smoothed)
    curr_track_data['FDI_calcs_smoothed'] = FDI_calcs
    curr_track_data['nucleus_zscore'] = nucleus_zscore
    curr_track_data['neigh_zscore'] = neigh_zscore
    curr_track_data['neigh_zscore_std'] = neigh_zscore_std


In [ ]:
track_lengths = {track_id: len(df) for track_id, df in all_tracks_processed_and_filtered.items()}
sorted_lengths = sorted(track_lengths.items(), key=lambda x: x[1], reverse=True)
for track_id, length in sorted_lengths:
    df = all_tracks_processed_and_filtered[track_id]
    if 'nucleus_zscore' in df.columns and 'neigh_zscore' in df.columns:
        final_idx = df.index[-1]
        final_nucleus_zscore = df.loc[final_idx, 'nucleus_zscore']
        final_neigh_zscore = df.loc[final_idx, 'neigh_zscore']
        print(f"Track ID: {track_id}, Length: {length}, Final nucleus_zscore: {final_nucleus_zscore}, Final neigh_zscore: {final_neigh_zscore}")
    else:
        print(f"Track ID: {track_id}, Length: {length}, Final nucleus_zscore: N/A, Final neigh_zscore: N/A")

In [ ]:
all_tracks_processed_and_filtered[3922]

In [ ]:
for track_id in sorted(all_tracks_processed_and_filtered.keys(), reverse=True):
    df = all_tracks_processed_and_filtered[track_id]
    final_idx = df.index[-1]
    final_nucleus_zscore = df.loc[final_idx, 'nucleus_zscore'] if 'nucleus_zscore' in df.columns else None
    final_neigh_zscore = df.loc[final_idx, 'neigh_zscore'] if 'neigh_zscore' in df.columns else None
    track_length = len(df)
    print(f"Track {track_id}: length={track_length}, final nucleus_zscore={final_nucleus_zscore}, final neigh_zscore={final_neigh_zscore}")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import numpy as np

index_track=3922
df = all_tracks_processed_and_filtered[index_track]
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(10, 13))

# Plot FDI calculations
sns.lineplot(data=df, x='time', y='FDI_calcs', linewidth=1, label='Raw', ax=ax1)
fdi_smoothed = savgol_filter(df['FDI_calcs'], window_length=15, polyorder=1)
sns.lineplot(data=df, x='time', y='FDI_calcs_smoothed', linewidth=1, label='Smoothed', ax=ax1)
ax1.set_xlabel('Time', fontsize=12)
ax1.set_ylabel('FDI Calculations', fontsize=12)
ax1.set_title('FDI Calculations vs Time for Track 4491', fontsize=14, pad=15)
ax1.legend()

# Plot Phan 2024 with mean
sns.lineplot(data=df, x='time', y='channel_2_norm', linewidth=2, label='Nucleus Normalized', ax=ax2)
sns.lineplot(data=df, x='time', y='neighs_mean_norm', linewidth=2, label='Neighbor Mean Normalized', ax=ax2)
ax2.set_xlabel('Time', fontsize=12)
ax2.set_ylabel('Normalized (Phan 2024) Values', fontsize=12)
ax2.set_title('Normalized (Mean) Values vs Time for Track 4491', fontsize=14, pad=15)
ax2.legend()

# Plot Phan 2024 with max
sns.lineplot(data=df, x='time', y='channel_2_norm', linewidth=2, label='Nucleus Normalized', ax=ax3)
sns.lineplot(data=df, x='time', y='maxs_norm', linewidth=2, label='Neighbor Max Normalized', ax=ax3)
ax3.set_xlabel('Time', fontsize=12)
ax3.set_ylabel('Normalized (Phan 2024) Values', fontsize=12)
ax3.set_title('Normalized (Max) Values vs Time for Track 4491', fontsize=14, pad=15)
ax3.legend()

# Plot z-scores relative to population
sns.lineplot(data=df, x='time', y='nucleus_zscore', linewidth=2, label='Nucleus Z-Score', ax=ax4)
sns.lineplot(data=df, x='time', y='neigh_zscore', linewidth=2, label='Neighbor Z-Score', ax=ax4)
ax4.set_xlabel('Time', fontsize=12)
ax4.set_ylabel('Population Z-Score', fontsize=12)
ax4.set_title('Population Z-Scores vs Time for Track 4491', fontsize=14, pad=15)
ax4.legend()

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

# --- plot lines and ribbon ---
ax.plot(df['time']*2.5 + 42*2.5, df['nucleus_zscore'], lw=2, marker='o', ms=4,
        label='Nucleus Z-Score', color='#FF6B6B')
ax.plot(df['time']*2.5 + 42*2.5, df['neigh_zscore'], lw=2, marker='s', ms=4,
        label='Neighbor Z-Score', color='#4ECDC4')
ax.fill_between(df['time']*2.5 + 42*2.5,
                df['neigh_zscore'] + df['neigh_zscore_std'],
                df['neigh_zscore'] - df['neigh_zscore_std'],
                alpha=0.2, color='#4ECDC4', label='Neighbor Z-Score ± STD')

for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(1)

ax.tick_params(axis='both', which='major', direction='in', length=6, width=1, colors='black')

for tick in ax.xaxis.get_major_ticks() + ax.xaxis.get_minor_ticks():
    tick.tick1line.set_clip_on(False)
    tick.tick2line.set_clip_on(False)
    tick.tick1line.set_visible(True)
    tick.tick2line.set_visible(True)

for tick in ax.yaxis.get_major_ticks() + ax.yaxis.get_minor_ticks():
    tick.tick1line.set_clip_on(False)
    tick.tick2line.set_clip_on(False)
    tick.tick1line.set_visible(True)
    tick.tick2line.set_visible(True)

ax.set_xlabel('Time', fontsize=12, color='black')
ax.set_ylabel('Population Z-Score', fontsize=12, color='black')
ax.set_title('Population Z-Scores with Uncertainty vs Time for Track 4491', fontsize=14, pad=15, color='black')
ax.legend()
# Save in multiple formats
# plt.savefig('zscore_uncertainty_plot.svg')
# plt.savefig('zscore_uncertainty_plot.pdf')
# plt.savefig('zscore_uncertainty_plot.png', dpi=300)
plt.show()

## Mutual Information Calculation: All Tracks

In [ ]:
window_size = 1
mi_estimate=np.zeros(310)

X_time_window={}
Y_time_window={}
Position_X_time_window={}
Position_Y_time_window={}
Position_Z_time_window={}
track_id_window={}
for time_now in range(1,311):
    X_time_window[time_now]=[]
    Y_time_window[time_now]=[]
    Position_X_time_window[time_now]=[]
    Position_Y_time_window[time_now]=[]
    Position_Z_time_window[time_now]=[]
    track_id_window[time_now]=[]

for track_id in all_tracks_processed_and_filtered.keys():
    print("Processing trackid: {}".format(track_id))
    curr_track_data_unfiltered=all_tracks_processed_and_filtered[track_id]
    curr_track_data = curr_track_data_unfiltered.dropna(subset=['channel_2_norm','neighs_mean_norm']).copy()
    time_range=curr_track_data['time'].unique()
    full_X=(curr_track_data['channel_2_norm']).to_numpy()
    full_Y=(curr_track_data['neighs_mean_norm']).to_numpy()

    # Plot min-max normalized values
    min_val = 0.0
    max_val = 1.0
    max_val_full_X=np.max(full_X)
    max_val_full_Y=np.max(full_Y)


    smoothed_FDI_calcs = curr_track_data['FDI_calcs_smoothed']
    flag_sop=False
    if track_id in sop_indicator:
        _, sb_index = sop_indicator[track_id]
        if sb_index>=1:
            flag_sop=True


    t_start=time_range[0]
    t_end=time_range[-1]
    for time_now in time_range:
        if time_now-window_size+1<t_start:
            continue
        time_now_window=[t_ for t_ in time_range if t_>=time_now-window_size+1 and t_<=time_now]
        if len(time_now_window)!=window_size:
            print("Window size is not correct for time {}".format(time_now))
            break
        curr_x_temp=curr_track_data['channel_2_norm'][curr_track_data['time'].isin(time_now_window)].to_numpy()
        curr_x = (curr_x_temp - min_val) / (max_val - min_val)
        X_time_window[time_now].append(list(curr_x))
        curr_y_temp=curr_track_data['neighs_mean_norm'][curr_track_data['time'].isin(time_now_window)].to_numpy()
        curr_y = (curr_y_temp - min_val) / (max_val - min_val)
        Y_time_window[time_now].append(list(curr_y))
        Position_X_time_window[time_now].append(list(curr_track_data['x'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        Position_Y_time_window[time_now].append(list(curr_track_data['y'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        Position_Z_time_window[time_now].append(list(curr_track_data['z'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        track_id_window[time_now].append(track_id)

time_window_lengths={}
for time_now in range(window_size,311):
    curr_window=X_time_window[time_now]
    time_window_lengths[time_now]=len(curr_window)

plt.figure(figsize=(12,8))
plt.plot(2.5*np.array(range(window_size,311)), list(time_window_lengths.values()), linewidth=2)
plt.xlabel('Time (minutes)', fontsize=12)
plt.ylabel('Number of tracks', fontsize=12)
plt.title(f'Number of tracks with window size {window_size} vs Time', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10)
plt.tight_layout()
plt.show()

In [ ]:
k_now = 4
window_size = 1

mi_estimate=np.zeros(310)

mi_estimate_window = []
for time_now in range(window_size, 311):
    X_now = X_time_window[time_now]
    Y_now = Y_time_window[time_now]
    if len(X_now)==0 or len(Y_now)==0:
        print("Oh nyo~")
        break
    mi_estimate_window.append(ee.mi(X_now,Y_now,k=k_now, base=np.exp(1)))

print(f"Maximum MI: {np.max(mi_estimate_window):.3f}")
print(f"Minimum MI: {np.min(mi_estimate_window):.3f}") 
print(f"Mean MI: {np.mean(mi_estimate_window):.3f}")

# Plot MI estimate
plt.figure(figsize=(16, 10))
plt.plot(2.5*np.array(range(window_size,311)), mi_estimate_window, linewidth=2, color='#2E86C1')
plt.ylabel('Mutual Information Estimate', fontsize=12)
plt.xlabel('Time (minutes)', fontsize=12)
plt.title('Mutual Information (Window Size = {}) Over Time (k={})'.format(window_size, k_now), fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10)
plt.show()


In [ ]:
k_now = 4
window_size = 1
mi_estimate=np.zeros(310)

X_time_window={}
Y_time_window={}
Position_X_time_window={}
Position_Y_time_window={}
Position_Z_time_window={}
track_id_window={}
for time_now in range(1,311):
    X_time_window[time_now]=[]
    Y_time_window[time_now]=[]
    Position_X_time_window[time_now]=[]
    Position_Y_time_window[time_now]=[]
    Position_Z_time_window[time_now]=[]
    track_id_window[time_now]=[]

for track_id in all_tracks_processed_and_filtered.keys():
    print("Processing trackid: {}".format(track_id))
    curr_track_data=all_tracks_processed_and_filtered[track_id]
    curr_track_data = curr_track_data.dropna(subset=['nucleus_zscore','neigh_zscore']).copy()
    time_range=curr_track_data['time'].unique()
    full_X=curr_track_data['nucleus_zscore'].to_numpy()
    full_Y=curr_track_data['neigh_zscore'].to_numpy()
    max_val_full_X=np.max(full_X)
    max_val_full_Y=np.max(full_Y)
    smoothed_FDI_calcs = curr_track_data['FDI_calcs_smoothed']
    flag_sop=False
    if track_id in sop_indicator:
        _, sb_index = sop_indicator[track_id]
        if sb_index>=1:
            flag_sop=True
    t_start=time_range[0]
    t_end=time_range[-1]
    for time_now in time_range:
        if time_now-window_size+1<t_start:
            continue
        time_now_window=[t_ for t_ in time_range if t_>=time_now-window_size+1 and t_<=time_now]
        if len(time_now_window)!=window_size:
            print("Window size is not correct for time {}".format(time_now))
            break
        curr_x=curr_track_data['nucleus_zscore'][curr_track_data['time'].isin(time_now_window)].to_numpy()
        X_time_window[time_now].append(list(curr_x))
        curr_y=curr_track_data['neigh_zscore'][curr_track_data['time'].isin(time_now_window)].to_numpy()
        Y_time_window[time_now].append(list(curr_y))
        Position_X_time_window[time_now].append(list(curr_track_data['x'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        Position_Y_time_window[time_now].append(list(curr_track_data['y'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        Position_Z_time_window[time_now].append(list(curr_track_data['z'][curr_track_data['time'].isin(time_now_window)].to_numpy()))
        track_id_window[time_now].append(track_id)

time_window_lengths={}
for time_now in range(window_size,311):
    curr_window=X_time_window[time_now]
    time_window_lengths[time_now]=len(curr_window)

plt.figure(figsize=(12,8))
plt.plot(2.5*np.array(range(window_size,311)), list(time_window_lengths.values()), linewidth=2)
plt.xlabel('Time (minutes)', fontsize=12)
plt.ylabel('Number of tracks', fontsize=12)
plt.title(f'Number of tracks with window size {window_size} vs Time', fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10)
plt.tight_layout()
plt.show()

mi_estimate_window = []
mi_shuffled_window = []
mi_shuffled_window_low = []
mi_shuffled_window_high = []
mi_jackknife_std = []  # For storing jackknife standard deviations
n_shuffles = 1000

for time_now in range(window_size, 311):
    print("Processing time: {}".format(time_now))
    X_now = X_time_window[time_now]
    Y_now = Y_time_window[time_now]
    if len(X_now)==0 or len(Y_now)==0:
        print("Oh nyo~")
        break
        
    # Calculate main MI estimate
    mi_val = ee.mi(X_now, Y_now, k=k_now, base=np.exp(1))
    mi_estimate_window.append(mi_val)
    
    # Jackknife resampling for standard deviation
    n = len(X_now)
    jackknife_estimates = []
    for i in range(n):
        # Leave one out
        X_jack = X_now[:i] + X_now[i+1:]
        Y_jack = Y_now[:i] + Y_now[i+1:]
        jack_mi = ee.mi(X_jack, Y_jack, k=k_now, base=np.exp(1))
        jackknife_estimates.append(jack_mi)
    
    # Calculate jackknife standard deviation
    jack_std = np.std(jackknife_estimates) * np.sqrt(n-1)
    mi_jackknife_std.append(jack_std)

    # Shuffling test
    shuffle_mean, (shuffle_low, shuffle_high)=ee.shuffle_test(ee.mi, X_now, Y_now, k=k_now, base=np.exp(1), ci=0.95, ns=n_shuffles)
    mi_shuffled_window.append(shuffle_mean)
    mi_shuffled_window_low.append(shuffle_low)
    mi_shuffled_window_high.append(shuffle_high)

print(f"Maximum MI: {np.max(mi_estimate_window):.3f}")
print(f"Minimum MI: {np.min(mi_estimate_window):.3f}") 
print(f"Mean MI: {np.mean(mi_estimate_window):.3f}")
print(f"\nMaximum shuffled MI: {np.max(mi_shuffled_window):.3f}")
print(f"Minimum shuffled MI: {np.min(mi_shuffled_window):.3f}")
print(f"Mean shuffled MI: {np.mean(mi_shuffled_window):.3f}")

# Plot MI estimate with error bars and ribbons
plt.figure(figsize=(16, 10))
time_points = 2.5*np.array(range(window_size,311))

# Plot shuffled MI and its confidence interval
plt.plot(time_points, mi_shuffled_window, linewidth=2, color='#E74C3C', linestyle='--', label='Shuffled MI')
plt.fill_between(time_points, mi_shuffled_window_low, mi_shuffled_window_high, alpha=0.2, color='#E74C3C')

# Plot actual MI with jackknife error bars and ribbon
plt.errorbar(time_points, mi_estimate_window, yerr=mi_jackknife_std, fmt='none', color='#2E86C1', alpha=0.3)
plt.plot(time_points, mi_estimate_window, linewidth=2, color='#2E86C1', label='Actual MI')
plt.fill_between(time_points, 
                 np.array(mi_estimate_window) - np.array(mi_jackknife_std),
                 np.array(mi_estimate_window) + np.array(mi_jackknife_std), 
                 alpha=0.2, color='#2E86C1')

plt.legend()
plt.ylabel('Mutual Information Estimate', fontsize=12)
plt.xlabel('Time (minutes)', fontsize=12)
plt.title('Mutual Information (Window Size = {}) Over Time (k={})'.format(window_size, k_now), fontsize=14, pad=15)
plt.grid(True, alpha=0.3)
plt.tick_params(axis='both', which='major', labelsize=10)
plt.show()


In [ ]:
times=2.5*np.array(range(window_size,311))
os.makedirs('MI_results', exist_ok=True)
np.save('MI_results/times_all_s3.npy', times)
np.save('MI_results/mi_estimate_window_all  _s3_zscore.npy', mi_estimate_window)
np.save('MI_results/mi_jackknife_std_all_s3_zscore.npy', mi_jackknife_std)
np.save('MI_results/mi_shuffled_window_all_s3_zscore.npy', mi_shuffled_window)
np.save('MI_results/mi_shuffled_window_all_last_s3_zscore.npy', mi_shuffled_window_low)
np.save('MI_results/mi_shuffled_window_all_last_s3_zscore.npy', mi_shuffled_window_high)

In [ ]:
times_to_plot = [1, 26, 51, 76, 151, 176, 201, 251, 301]
fig, axes = plt.subplots(3, 3, figsize=(18, 12))
fig.suptitle('Nucleus vs Neighbor Intensity Correlation Over Time', fontsize=16, y=0.98)
axes = axes.flatten()

# First find global min/max to set consistent axes
x_min, x_max = float('inf'), float('-inf')
y_min, y_max = float('inf'), float('-inf')
for t in times_to_plot:
    curr_X = np.array(X_time_window[t]).flatten()
    curr_Y = np.array(Y_time_window[t]).flatten()
    x_min = min(x_min, curr_X.min())
    x_max = max(x_max, curr_X.max())
    y_min = min(y_min, curr_Y.min())
    y_max = max(y_max, curr_Y.max())

for i, t in enumerate(times_to_plot):
    curr_X = X_time_window[t]
    curr_Y = Y_time_window[t]
    
    df = pd.DataFrame({
        'Nucleus Intensity': np.array(curr_X).flatten(),
        'Neighbor Intensity': np.array(curr_Y).flatten()
    })
    
    # Create scatter plot with orange color and transparency
    sns.scatterplot(data=df, x='Nucleus Intensity', y='Neighbor Intensity', 
                   alpha=0.4, ax=axes[i], color='#d36027')
    
    # # Calculate and display MI
    # mi = ee.mi(curr_X, curr_Y, k=k_now, base=np.exp(1))
    # axes[i].text(0.05, 0.95, f'MI = {mi:.3f}', 
    #             transform=axes[i].transAxes, fontsize=10)
    
    axes[i].set_title(f't = {(t-1)*2.5} minutes', fontsize=12)
    axes[i].grid(True, alpha=0.8)
    axes[i].tick_params(axis='both', which='major', labelsize=10)
    sns.despine(ax=axes[i])
    
    # Set same axis limits for all subplots
    axes[i].set_xlim([x_min, x_max])
    axes[i].set_ylim([y_min, y_max])

plt.tight_layout()
plt.savefig('MI_results/'+name_exp+'_scatter_plots_not_shuffle.png', dpi=300, bbox_inches='tight')
plt.savefig('MI_results/'+name_exp+'_scatter_plots_not_shuffle.svg', dpi=300, bbox_inches='tight')
plt.savefig('MI_results/'+name_exp+'_scatter_plots_not_shuffle.pdf', dpi=300, bbox_inches='tight')
plt.show()
